# NAQSHKASH Stage 1 — Kaggle GPU preflight

Enable a GPU accelerator in notebook settings, attach private code and Stage 1 datasets, then set the three paths below. Kaggle may expose the uploaded ZIP contents directly under /kaggle/input. Read-only project/data paths work; outputs must be writable under /kaggle/working.

Run cells sequentially. Run All performs probes and only 40 controlled training batches; **full 30-epoch training and normal resume commands are printed, never executed.** Use a fresh OUTPUT_ROOT for a new smoke test. Python 3.8+, PyTorch 2.3+ required.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

# Edit these three paths. Keep code, dataset and run outputs separate.
PROJECT_ROOT = Path('/kaggle/input/your-code-dataset/NAQSHKASH')
DATASET_ROOT = Path('/kaggle/input/your-stage1-dataset')
OUTPUT_ROOT = Path('/kaggle/working/naqshkash')

# Initial candidate only; lower manually on OOM. Never increased automatically.
WIDTH_BUDGET = 16384
MAX_BATCH_SIZE = 32
NUM_WORKERS = 2

# Single GPU even when the platform offers two. Run from a fresh kernel if torch was already imported.
if 'torch' in sys.modules:
    raise RuntimeError('Restart the kernel and run this configuration cell before importing torch')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'


## Environment and GPU gate

This notebook stops if CUDA is unavailable; the trainer itself also supports CPU.

In [ ]:
import torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('PyTorch CUDA version:', torch.version.cuda)
assert tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) >= (2, 3), 'PyTorch 2.3+ required'
assert torch.cuda.is_available(), 'CUDA unavailable: enable a GPU accelerator/runtime and restart the kernel'
assert torch.cuda.device_count() == 1, 'Use one visible GPU'
gpu = torch.cuda.get_device_properties(0)
print('GPU:', gpu.name)
print('GPU total memory GiB:', gpu.total_memory / 2**30)
free, total = torch.cuda.mem_get_info()
print('GPU free memory GiB:', free / 2**30)
print('AMP GradScaler enabled:', torch.amp.GradScaler('cuda').is_enabled())
if __import__('shutil').which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=True)


## Check paths and install only missing dependencies

Do not reinstall the platform CUDA PyTorch. If a dependency is missing, Internet access is needed for pip. If DATASET_ROOT still contains only a ZIP, see the deployment guide: the current dataset reader requires extracted files.

In [ ]:
for folder in ('training', 'models', 'data', 'tests'):
    assert (PROJECT_ROOT / folder).is_dir(), f'Missing project directory: {PROJECT_ROOT / folder}'
assert (PROJECT_ROOT / 'training/train_crnn.py').is_file(), 'PROJECT_ROOT must be the repository root'
for split in ('train', 'validation', 'test'):
    for item in ('images', 'labels', 'metadata.csv'):
        assert (DATASET_ROOT / split / item).exists(), f'Missing dataset path: {DATASET_ROOT / split / item}'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import importlib.util
packages = {'numpy': 'numpy', 'cv2': 'opencv-python', 'scipy': 'scipy', 'pytest': 'pytest'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
else:
    print('Dependencies already present; no installation needed')
from training.gpu_deployment import commands, display_command, inspect_checkpoints, verify_layout
verify_layout(PROJECT_ROOT, DATASET_ROOT)
subprocess.run([sys.executable, '-m', 'training.train_crnn', '--help'], check=True)
COMMANDS = commands(DATASET_ROOT, OUTPUT_ROOT, WIDTH_BUDGET, MAX_BATCH_SIZE, NUM_WORKERS)

def run_short(name):
    assert name in ('preflight', 'interrupt', 'resume_test', 'full_width_probe')
    assert torch.cuda.is_available(), 'GPU disappeared; enable GPU and restart'
    print(display_command(COMMANDS[name]), flush=True)
    subprocess.run(COMMANDS[name], cwd=PROJECT_ROOT, check=True)


## Short GPU memory preflight

Uses 4096 TRAIN / 256 validation samples, AMP and width-aware batching. It probes high-memory batches within that subset, performs no epoch validation/test and creates **no .pt checkpoint**. The trainer still scans all TRAIN labels for its exact vocabulary and indexes full metadata before applying limits. Initial file scanning can take minutes with 1.3 million files; the GPU computation is only a few batches. A mandatory full-width probe below checks outliers outside this subset.

In [ ]:
run_short('preflight')
print('Memory-only probe completed; checkpoints are created by the next test, not --preflight-only.')


## Deliberate interruption and checkpoint inspection

20 batches, saving every 10; stops before validation/test. The test directory must be fresh. best.pt is expected to be absent until epoch validation selects a model.

In [ ]:
SMOKE_ROOT = OUTPUT_ROOT / 'resume_test'
assert not (SMOKE_ROOT / 'last.pt').exists(), 'Existing smoke checkpoint: choose a fresh OUTPUT_ROOT or continue with the resume cell'
run_short('interrupt')
before = inspect_checkpoints(SMOKE_ROOT)
assert before is not None, 'Missing last.pt: check the training logs and output path'
assert before['epoch'] == 0 and before['next_batch'] == 20 and before['global_step'] == 20
assert before['scaler_active'], 'CUDA AMP scaler state missing'


## Resume for 20 more batches

The exact same limits, seed, width budget, maximum batch size and AMP setting are retained. Look for Resume: epoch=0 next_batch=20 global_step=20 in logs; then saved counters must reach 40. Checkpoint inspection checks continuation automatically.

In [ ]:
if 'before' not in globals():
    before = inspect_checkpoints(OUTPUT_ROOT / 'resume_test')
    assert before is not None, 'Restore last.pt into resume_test before running this cell'
run_short('resume_test')
after = inspect_checkpoints(OUTPUT_ROOT / 'resume_test')
assert after['epoch'] == before['epoch'] == 0
assert after['next_batch'] == before['next_batch'] + 20
assert after['global_step'] == before['global_step'] + 20
assert after['scaler_active']
print('PASS: checkpoint/scaler present and resumed batch/global step continued')


## Full TRAIN width coverage before a real run

No TRAIN limit: reconstructs the full epoch plan and probes its largest-area and longest-sequence batches. No training epoch, validation or test runs. This first full scan takes longer than the subset probe. Do not treat a passing subset probe as full-dataset memory calibration. 16384 is a candidate, not a measured safe T4 budget; leave headroom in printed memory diagnostics.

In [ ]:
run_short('full_width_probe')
print('GPU probes and controlled resume test passed. Full training is still a manual action.')


## Full training and normal resume — DISPLAY ONLY

Use a new full_run directory for full training. The smoke checkpoint uses subset limits and cannot be resumed as a full run. Copy the appropriate printed command into a separate terminal/cell only when you choose to train. No limits, no skip-test: full epoch validation and final test retain current trainer behavior.

In [ ]:
print('FULL TRAINING (not executed):\n' + display_command(COMMANDS['full']))
print('NORMAL RESUME (not executed):\n' + display_command(COMMANDS['resume']))


## Preserve outputs before runtime loss

Keep run files under /kaggle/working. Download last.pt and best.pt (when present), history and configuration notes, or save a notebook version with outputs included and verify that its Output contains them. Saved notebook code alone does not prove outputs are preserved. A new session does not automatically restore them. Attach saved outputs as input and copy just the matching checkpoint pair/history into a writable full_run directory before resume. Never use Run All as a way to start the full command: those cells only print it.

Only last.pt and best.pt are maintained, plus temporary atomic-write files. Keep matching files from one stopped run. Do not mix smoke and full checkpoints. See training/GPU_DEPLOYMENT.md for ZIP extraction, persistence, and error recovery.